# FREUID — VESSL Cloud GPU training

Colab 버전(`run_baseline.ipynb`)과 달리 **ngrok 터널 / Drive 마운트가 필요 없다.**
VESSL Workspace가 파일과 환경을 유지해주기 때문.

## 최초 1회 세팅 (VESSL Workspace 생성 직후)

1. cloud.vessl.ai → New Workspace → GPU 선택 (A100 권장)
2. Image: `vessl/ml:latest` 또는 `pytorch/pytorch:2.x-cuda12.x-cudnn9-devel`
3. JupyterLab 또는 VS Code SSH로 접속
4. 아래 STEP 1 셀들 순서대로 실행

> **Pause & Resume**: VESSL은 Workspace를 일시정지해도 파일이 유지된다.
> checkpoints/, submissions/ 폴더가 그대로 남아있으므로 Drive 마운트 불필요.

## STEP 1 — 환경 세팅 (최초 1회)

In [ ]:
# GPU 확인
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu  <- GPU 설정 확인 필요')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('cwd:', os.getcwd())

In [ ]:
# 레포 클론 + 패키지 설치 (최초 1회)
# 이미 클론돼 있으면 스킵
import os
if not os.path.exists('kaggle_freuid_2026'):
    !git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
    %cd kaggle_freuid_2026
    !pip install -q -e .
else:
    %cd kaggle_freuid_2026
    print('이미 클론됨, 패키지 재설치')
    !pip install -q -e .

In [ ]:
# Kaggle 인증 세팅
# 방법 A: VESSL 환경변수로 주입 (VESSL Dashboard -> Workspace -> Environment Variables)
#   KAGGLE_USERNAME=xxx  KAGGLE_KEY=yyy
# 방법 B: 아래 직접 입력 (보안 주의 — 공유 금지)
import os, json

username = os.environ.get('KAGGLE_USERNAME', '')
key      = os.environ.get('KAGGLE_KEY', '')

if not username or not key:
    # 환경변수 없으면 직접 입력 (이 셀 실행 후 값 지울 것)
    username = 'YOUR_KAGGLE_USERNAME'
    key      = 'YOUR_KAGGLE_KEY'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json 완료')

In [ ]:
# 데이터 다운로드 (최초 1회, ~수 GB)
# VESSL Workspace는 스토리지가 유지되므로 한 번만 받으면 됨
import os
if not os.path.exists('data/train'):
    !kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
    !cd data && unzip -q -o '*.zip' && rm -f *.zip
    print('다운로드 완료')
else:
    print('데이터 이미 존재 — 스킵')

!echo 'train imgs:' && ls data/train/train | wc -l
!ls data

## STEP 2 — 학습

In [ ]:
# ViT baseline (cross_domain val)
CONFIG = 'configs/a100_vit_all_strategies.yaml'
!python -m freuid.train --config {CONFIG}

In [ ]:
# DCT 듀얼 스트림 (RGB + 주파수 branch)
# 로그에 'model=DualStream(RGB+DCT)' 뜨면 정상
CONFIG = 'configs/a100_vit_dct.yaml'
!python -m freuid.train --config {CONFIG}

In [ ]:
# LOO K-Fold (특정 fold만 실행 — 전체는 아래 루프 사용)
CONFIG = 'configs/a100_vit_all_strategies.yaml'
FOLD   = 0   # 0~6 중 선택
!python -m freuid.train --config {CONFIG} --fold {FOLD}

In [ ]:
# LOO 전체 fold 순서대로 실행
CONFIG = 'configs/a100_vit_all_strategies.yaml'
for fold in range(7):
    print(f'\n{'='*50}')
    print(f'LOO fold {fold}/6')
    print('='*50)
    !python -m freuid.train --config {CONFIG} --fold {fold}

## STEP 3 — Inference & 제출

In [ ]:
# ViT baseline inference
CKPT = 'checkpoints/a100_vit_all_strategies.pt'
OUT  = 'submissions/a100_vit_all_strategies.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT} --tta 5
!head -3 {OUT} && wc -l {OUT}

In [ ]:
# DCT 모델 inference
CKPT = 'checkpoints/a100_vit_dct.pt'
OUT  = 'submissions/a100_vit_dct.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT} --tta 5
!head -3 {OUT} && wc -l {OUT}

In [ ]:
# ViT baseline 제출
OUT = 'submissions/a100_vit_all_strategies.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'vit cross_domain'

In [ ]:
# DCT 듀얼 스트림 제출
OUT = 'submissions/a100_vit_dct.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'vit+dct dual stream tta5'

## 참고 — Colab과의 차이점

| | Colab | VESSL |
|---|---|---|
| 터널 | ngrok 필요 | SSH/JupyterLab 직접 제공 |
| 파일 유지 | Drive 마운트 필요 | Workspace 자동 유지 |
| 세션 제한 | 90분 idle / 12h max | 크레딧 소진 전까지 유지 |
| Kaggle 인증 | Colab Secrets | 환경변수 또는 직접 입력 |
| Resume 학습 | Drive 통해 체크포인트 | 로컬에 그대로 존재 |

**VESSL에서 학습 중단 후 재개:**
```bash
# _resume.pt 가 자동으로 감지되어 이어서 학습됨
python -m freuid.train --config configs/a100_vit_all_strategies.yaml
```